# TP7 — Dashboard MongoDB de supervision des accès
Ce notebook est le dashboard TP7. Il extrait les données de MongoDB pour continuer à fonctionner lorsque le site Flask est mis en ligne.


In [ ]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pymongo import MongoClient
from spark_jobs.risk_scoring import score_dataframe_pandas, top_risky_users
load_dotenv(ROOT / ".env")


In [ ]:
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
DB_NAME = os.getenv("DB_NAME", "hopital_db")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
db = client[DB_NAME]
client.admin.command("ping")
print(f"MongoDB connecté : {DB_NAME}")


In [ ]:
def read_collection(name):
    """Lit une collection MongoDB sans le champ technique _id."""
    return pd.DataFrame(list(db[name].find({}, {"_id": 0})))

alerts = read_collection("alerts")
if not alerts.empty:
    dashboard_df = alerts.copy()
    source = "MongoDB.alerts"
else:
    access_logs = read_collection("access_logs")
    source = "MongoDB.access_logs"
    if access_logs.empty:
        access_logs = pd.read_csv(ROOT / "datasets" / "access_logs.csv")
        source = "datasets/access_logs.csv (secours local)"
    access_logs["timestamp"] = pd.to_datetime(access_logs["timestamp"], errors="coerce")
    access_logs["success"] = access_logs["success"].astype(str).str.lower().isin(["true", "1", "yes", "oui"])
    access_logs["mfa_passed"] = access_logs["mfa_passed"].astype(str).str.lower().isin(["true", "1", "yes", "oui"])
    if "bytes" in access_logs.columns:
        access_logs["bytes"] = pd.to_numeric(access_logs["bytes"], errors="coerce").fillna(0).astype(int)
    access_logs["hour"] = access_logs["timestamp"].dt.hour
    dashboard_df = score_dataframe_pandas(access_logs)
    dashboard_df = dashboard_df[dashboard_df["risk_score"] >= 6].copy()
print(f"Source : {source} — {len(dashboard_df)} alertes élevées")
dashboard_df.head()


In [ ]:
# Minimisation : aucune donnée patient sensible n’est chargée ni affichée.
allowed_columns = [
    "timestamp", "user_id", "role", "department", "resource_id", "resource_type",
    "sensitivity", "action", "ip", "success", "mfa_passed", "risk_score",
    "risk_level", "risk_reasons", "alert_id",
]
dashboard_df = dashboard_df[[column for column in allowed_columns if column in dashboard_df.columns]].copy()
dashboard_df["timestamp"] = pd.to_datetime(dashboard_df["timestamp"], errors="coerce")
dashboard_df["hour"] = dashboard_df["timestamp"].dt.hour
dashboard_df.head()


In [ ]:
top10 = pd.DataFrame(top_risky_users(dashboard_df.to_dict(orient="records"), 10))
top10


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dashboard_df.groupby("hour").size().sort_index().plot(kind="bar", ax=axes[0], color="firebrick", title="Alertes par heure")
axes[0].set_xlabel("Heure")
axes[0].set_ylabel("Nombre d’alertes")
dashboard_df.groupby("action").size().sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="steelblue", title="Alertes par action")
axes[1].set_xlabel("Action")
axes[1].set_ylabel("Nombre d’alertes")
plt.tight_layout()


In [ ]:
if top10.empty:
    interpretation = "Aucune alerte élevée disponible dans MongoDB."
else:
    leader = top10.iloc[0]
    interpretation = (
        f"Priorité SOC : {leader['user_id']} cumule {leader['nb_alertes']} alertes "
        f"et un score total de {leader['total_score']}."
    )
interpretation


## Configuration de déploiement
- Renseigner `MONGO_URI` et `DB_NAME` dans l’environnement de production.
- Alimenter `access_logs` et `alerts` dans la même base MongoDB que le site.
- Garder `alerts` comme source prioritaire du dashboard notebook.
- Ne pas charger `patients_sensibles.csv` dans le dashboard.
